In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)

categorical = [c for c in df.columns if df[c].dtype == 'object' or str(df[c].dtype) == 'str']
numerical = [c for c in df.columns if c not in categorical and c != 'converted']
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
y_train, y_val = df_train.converted.values, df_val.converted.values
features = categorical + numerical

In [2]:
for c in ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']:
    auc = roc_auc_score(y_train, df_train[c])
    if auc < 0.5:
        auc = roc_auc_score(y_train, -df_train[c])
    print(c, round(auc, 4))

lead_score 0.7882
number_of_courses_viewed 0.723
interaction_count 0.7649
annual_income 0.6082


In [3]:
def train(d, C=1.0):
    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(d[features].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000).fit(X, d.converted.values)
    return dv, model

def predict(d, dv, model):
    return model.predict_proba(dv.transform(d[features].to_dict(orient='records')))[:, 1]

dv, model = train(df_train)
y_pred = predict(df_val, dv, model)
print(round(roc_auc_score(y_val, y_pred), 3))

0.732


In [4]:
rows = []
for t in np.arange(0, 1.01, 0.01):
    pred = y_pred >= t
    tp = (pred & (y_val == 1)).sum()
    fp = (pred & (y_val == 0)).sum()
    fn = (~pred & (y_val == 1)).sum()
    p = tp / (tp + fp) if tp + fp else 0
    r = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * p * r / (p + r) if p + r else 0
    rows.append((round(t, 2), p, r, f1))
scores = pd.DataFrame(rows, columns=['threshold', 'precision', 'recall', 'f1'])

valid = scores[(scores.precision > 0) | (scores.recall > 0)]
print(valid.loc[(valid.precision - valid.recall).abs().idxmin(), 'threshold'])

0.63


In [5]:
print(scores.loc[scores.f1.idxmax(), 'threshold'])

0.41


In [6]:
kfold = KFold(n_splits=5, shuffle=True, random_state=1)

def cross_val(C=1.0):
    aucs = []
    for tr, va in kfold.split(df_full_train):
        d_tr, d_va = df_full_train.iloc[tr], df_full_train.iloc[va]
        dv_, m_ = train(d_tr, C)
        aucs.append(roc_auc_score(d_va.converted.values, predict(d_va, dv_, m_)))
    return np.mean(aucs), np.std(aucs)

print(round(cross_val(1.0)[1], 3))

0.007


In [7]:
for C in [0.000001, 0.001, 1]:
    mean, std = cross_val(C)
    print(C, round(mean, 3), round(std, 3))

1e-06 0.617 0.019
0.001 0.749 0.01
1 0.732 0.007
